# Bài 1 v4 · Vector và độ tương đồng cosine
PGS.TS. Nguyễn Văn Hậu

Mục tiêu: đọc dữ liệu nguồn, dùng cùng schema, tính và diễn giải có điều kiện. Các điểm ở đây đo nhãn metadata; chưa đánh giá sở thích người dùng. Mở notebook trong thư mục Bai01_v4. Cài `numpy`, `scikit-learn` và `jupyter` theo requirements.txt nếu môi trường chưa có.


In [ ]:
from pathlib import Path
import sys, math
import numpy as np

ROOT = Path.cwd()
if not (ROOT / "source/cosine_lab.py").exists():
    raise RuntimeError("Mở notebook tại thư mục Bai01_v4 chứa source/ và data/.")
sys.path.insert(0, str(ROOT / "source"))
from cosine_lab import load_movies, labels, encode, cosine_safe, AXES3, QUERY, CANDIDATES
from sklearn.metrics.pairwise import cosine_similarity
movies = load_movies()
print("Số phim:", len(movies))
print("Ba trục:", AXES3)
print("Dòng nguồn:", movies[QUERY])


Số phim: 9742
Ba trục: ('Action', 'Comedy', 'Sci-Fi')
Dòng nguồn: {'movieId': '2571', 'title': 'Matrix, The (1999)', 'genres': 'Action|Sci-Fi|Thriller'}


## 1. Dữ liệu → vector
Mỗi vị trí trả lời về một nhãn. Dùng chung AXES3 cho mọi hàng; movieId là khóa tra cứu. Ma trận X có 5 hàng ứng viên × 3 cột nhãn, còn u là vector tham chiếu có 3 thành phần.


In [ ]:
u = encode(movies[QUERY], AXES3)
X = np.stack([encode(movies[mid], AXES3) for mid in CANDIDATES])
print("u =", u.astype(int))
print("X (5 × 3) =\n", X.astype(int))
for mid, v in zip(CANDIDATES, X):
    print(mid, movies[mid]["title"], "→", v.astype(int))
assert X.shape == (5, 3)


u = [1 0 1]
X (5 × 3) =
 [[1 0 1]
 [1 1 1]
 [1 0 1]
 [0 0 1]
 [0 1 0]]
260 Star Wars: Episode IV - A New Hope (1977) → [1 0 1]
1580 Men in Black (a.k.a. MIB) (1997) → [1 1 1]
480 Jurassic Park (1993) → [1 0 1]
1214 Alien (1979) → [0 0 1]
1 Toy Story (1995) → [0 1 0]


## 2. Một hàng tính có bước
Giữ căn chính xác ở phép tính tay; trong mã so kết quả trước làm tròn với sai số 10⁻¹². Zero là không xác định, khác điểm 0 có chuẩn dương.


In [ ]:
v = X[1]  # Men in Black
dot = float(u @ v)
nu, nv = float(np.linalg.norm(u)), float(np.linalg.norm(v))
print("dot =", dot, "; chuẩn u =", nu, "; chuẩn v =", nv)
hand = dot / math.sqrt(float(u @ u) * float(v @ v))
own = cosine_safe(u, v)
lib = float(cosine_similarity([u], [v])[0, 0])
print("Tính tay / NumPy / scikit-learn:", hand, own, lib)
assert abs(hand - own) < 1e-12 and abs(own - lib) < 1e-12
for mid, v in zip(CANDIDATES, X):
    print(movies[mid]["title"], "dot", int(u @ v), "cos", format(cosine_safe(u, v), ".4f"))
assert sum(int(u @ v) for v in X) == 7


dot = 2.0 ; chuẩn u = 1.4142135623730951 ; chuẩn v = 1.7320508075688772
Tính tay / NumPy / scikit-learn: 0.8164965809277261 0.816496580927726 0.816496580927726
Star Wars: Episode IV - A New Hope (1977) dot 2 cos 1.0000
Men in Black (a.k.a. MIB) (1997) dot 2 cos 0.8165
Jurassic Park (1993) dot 2 cos 1.0000
Alien (1979) dot 1 cos 0.7071
Toy Story (1995) dot 0 cos 0.0000


## 3. Điều kiện và phản ví dụ
Các mảng cùng kích thước chưa đảm bảo cùng nghĩa trục. Hàm biết kích thước và số hữu hạn; người gọi phải dùng encoder chung. scikit-learn trả 0 ở vector 0 theo quy ước triển khai, trong khi hàm của bài giữ None.


In [ ]:
print("Toy Story:", cosine_safe(u, [0, 1, 0]))
print("Vector 0:", cosine_safe(u, [0, 0, 0]))
print("Quy ước thư viện:", cosine_similarity([u], [[0, 0, 0]])[0, 0])
assert cosine_safe(u, [0, 1, 0]) == 0
assert cosine_safe(u, [0, 0, 0]) is None
assert abs(cosine_safe(3*u, X[1]) - cosine_safe(u, X[1])) < 1e-12
for a, b in [([1, 0], [0, 1]), ([1, 0, 1], [0, 1, 1]), ([1], [1]), ([1, 0], [1, 1])]:
    print(a, b, "→", cosine_safe(a, b))


Toy Story: 0.0
Vector 0: None
Quy ước thư viện: 0.0
[1, 0] [0, 1] → 0.0
[1, 0, 1] [0, 1, 1] → 0.4999999999999999
[1] [1] → 1.0
[1, 0] [1, 1] → 0.7071067811865475


## 4. Giữ phim, thay biểu diễn
Lược đồ đầy đủ có 19 nhãn metadata, gồm IMAX. Ký hiệu `(no genres listed)` chỉ tình trạng thiếu nhãn, không tạo một tọa độ thể loại. Giao tập nhãn cung cấp một cách kiểm độc lập cho vector multi-hot.


In [ ]:
axes_full = sorted(set().union(*(labels(row) for row in movies.values())))
uf = encode(movies[QUERY], axes_full)
print("19 nhãn:", axes_full)
for mid in CANDIDATES:
    vf = encode(movies[mid], axes_full)
    a, b = labels(movies[QUERY]), labels(movies[mid])
    independent = len(a & b) / math.sqrt(len(a) * len(b))
    full = cosine_safe(uf, vf)
    assert abs(full - independent) < 1e-12
    print(movies[mid]["title"], "3 nhãn", round(cosine_safe(u, encode(movies[mid], AXES3)), 4), "19 nhãn", round(full, 4))
print("Casablanca: 3 nhãn", cosine_safe(u, encode(movies[912], AXES3)), "đủ nhãn", cosine_safe(uf, encode(movies[912], axes_full)))


19 nhãn: ['Action', 'Adventure', 'Animation', 'Children', 'Comedy', 'Crime', 'Documentary', 'Drama', 'Fantasy', 'Film-Noir', 'Horror', 'IMAX', 'Musical', 'Mystery', 'Romance', 'Sci-Fi', 'Thriller', 'War', 'Western']
Star Wars: Episode IV - A New Hope (1977) 3 nhãn 1.0 19 nhãn 0.6667
Men in Black (a.k.a. MIB) (1997) 3 nhãn 0.8165 19 nhãn 0.6667
Jurassic Park (1993) 3 nhãn 1.0 19 nhãn 0.866
Alien (1979) 3 nhãn 0.7071 19 nhãn 0.4082
Toy Story (1995) 3 nhãn 0.0 19 nhãn 0.0
Casablanca: 3 nhãn None đủ nhãn 0.0


## 5. Vận dụng và tự chấm
Trước khi chạy ô đáp án, tự tính cho v=(0;1;1), giải thích giới hạn kết luận, xét 3u và v=0. Rubric: mã hóa/trục 2 điểm, phép tính 3, diễn giải/hệ số dương 3, vector 0 2. Bài tập: chọn hai movieId khác; lập bảng 3 nhãn/19 nhãn, một hàng tính tay và nhận xét ≤150 từ.


In [ ]:
v = np.array([0., 1., 1.])
print("dot", u @ v, "; chuẩn", np.linalg.norm(u), np.linalg.norm(v))
print("cos", cosine_safe(u, v), "; 3u", cosine_safe(3*u, v), "; zero", cosine_safe(u, np.zeros(3)))
assert abs(cosine_safe(u, v) - .5) < 1e-12
from cosine_lab import run
report = run()
print("Kiểm toàn kho:", report["candidate_pool"], "ứng viên;", report["exact_equal_label_count"], "phim khác cùng tập nhãn")


dot 1.0 ; chuẩn 1.4142135623730951 1.4142135623730951
cos 0.4999999999999999 ; 3u 0.5 ; zero None
{"status": "pass", "checks": 38, "candidate_pool": 9707, "maximum_error": 2.220446049250313e-16}
Kiểm toàn kho: 9707 ứng viên; 42 phim khác cùng tập nhãn


In [ ]:
# yêu cầu 1
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

# Tải dữ liệu MovieLens
url = "https://files.grouplens.org/datasets/movielens/ml-latest-small.zip"

import requests
import zipfile
import io

response = requests.get(url)
zip_file = zipfile.ZipFile(io.BytesIO(response.content))

movies = pd.read_csv(
    zip_file.open("ml-latest-small/movies.csv")
)

# Hiển thị một số dòng dữ liệu
movies.head()

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


In [ ]:
# Phim tham chiếu
reference_title = "Toy Story (1995)"

# Hai phim cần so sánh
candidate_titles = [
    "Men in Black (a.k.a. MIB) (1997)",
    "Jurassic Park (1993)"
]

# Tìm thông tin phim
selected_titles = [reference_title] + candidate_titles

selected_movies = movies[
    movies["title"].isin(selected_titles)
]

print(selected_movies[["movieId", "title", "genres"]].to_string(index=False))

 movieId                            title                                      genres
       1                 Toy Story (1995) Adventure|Animation|Children|Comedy|Fantasy
     480             Jurassic Park (1993)            Action|Adventure|Sci-Fi|Thriller
    1580 Men in Black (a.k.a. MIB) (1997)                        Action|Comedy|Sci-Fi


Nguồn: [GroupLens](https://grouplens.org/datasets/movielens/latest/), bản movies.csv đi kèm có SHA-256; [cosine_similarity](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.pairwise.cosine_similarity.html). Chạy mã xác nhận phép tính, chưa xác nhận tính hữu ích của một hệ gợi ý.


In [ ]:
# Ba nhãn đặc trưng
axes_3 = ["Action", "Comedy", "Sci-Fi"]

def encode_movie(genres, axes):
    genre_set = set(genres.split("|"))
    return np.array(
        [int(genre in genre_set) for genre in axes]
    )

# Lấy dữ liệu của từng phim
movie_data = selected_movies.set_index("title")

# Tạo vector phim tham chiếu
u = encode_movie(
    movie_data.loc[reference_title, "genres"],
    axes_3
)

print("Phim tham chiếu:", reference_title)
print("Vector u:", u)

# Tạo vector cho hai phim so sánh
for title in candidate_titles:
    v = encode_movie(
        movie_data.loc[title, "genres"],
        axes_3
    )
    print(title, ":", v)

Phim tham chiếu: Toy Story (1995)
Vector u: [0 1 0]
Men in Black (a.k.a. MIB) (1997) : [1 1 1]
Jurassic Park (1993) : [1 0 1]


In [ ]:
results_3 = []

for title in candidate_titles:
    v = encode_movie(
        movie_data.loc[title, "genres"],
        axes_3
    )

    similarity = cosine_similarity(
        u.reshape(1, -1),
        v.reshape(1, -1)
    )[0, 0]

    results_3.append({
        "Phim so sánh": title,
        "Vector 3 nhãn": v.tolist(),
        "Cosine (3 nhãn)": round(similarity, 4)
    })

df_3 = pd.DataFrame(results_3)
df_3

,Phim so sánh,Vector 3 nhãn,Cosine (3 nhãn)
0,Men in Black (a.k.a. MIB) (1997),"[1, 1, 1]",0.5774
1,Jurassic Park (1993),"[1, 0, 1]",0.0000


In [ ]:
# Lấy toàn bộ nhãn thể loại, không tính nhãn thiếu
all_genres = sorted({
    genre
    for genre_string in movies["genres"]
    for genre in genre_string.split("|")
    if genre != "(no genres listed)"
})

print("Số nhãn:", len(all_genres))
print("Các nhãn:", all_genres)

# Tạo vector 19 nhãn cho phim tham chiếu
u_full = encode_movie(
    movie_data.loc[reference_title, "genres"],
    all_genres
)

results_19 = []

for title in candidate_titles:
    v_full = encode_movie(
        movie_data.loc[title, "genres"],
        all_genres
    )

    similarity = cosine_similarity(
        u_full.reshape(1, -1),
        v_full.reshape(1, -1)
    )[0, 0]

    results_19.append({
        "Phim so sánh": title,
        "Cosine (19 nhãn)": round(similarity, 4)
    })

df_19 = pd.DataFrame(results_19)
df_19

Số nhãn: 19
Các nhãn: ['Action', 'Adventure', 'Animation', 'Children', 'Comedy', 'Crime', 'Documentary', 'Drama', 'Fantasy', 'Film-Noir', 'Horror', 'IMAX', 'Musical', 'Mystery', 'Romance', 'Sci-Fi', 'Thriller', 'War', 'Western']


,Phim so sánh,Cosine (19 nhãn)
0,Men in Black (a.k.a. MIB) (1997),0.2582
1,Jurassic Park (1993),0.2236


In [ ]:
final_result = df_3.merge(
    df_19,
    on="Phim so sánh"
)

print("Phim tham chiếu:", reference_title)
display(final_result)

# Lưu bảng kết quả thành file CSV
final_result.to_csv(
    "ket_qua_cosine.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Đã lưu file ket_qua_cosine.csv")

Phim tham chiếu: Toy Story (1995)


,Phim so sánh,Vector 3 nhãn,Cosine (3 nhãn),Cosine (19 nhãn)
0,Men in Black (a.k.a. MIB) (1997),"[1, 1, 1]",0.5774,0.2582
1,Jurassic Park (1993),"[1, 0, 1]",0.0000,0.2236


Đã lưu file ket_qua_cosine.csv


In [ ]:
for _, row in final_result.iterrows():
    print(f"\nPhim: {row['Phim so sánh']}")
    print(f"Cosine với 3 nhãn: {row['Cosine (3 nhãn)']}")
    print(f"Cosine với 19 nhãn: {row['Cosine (19 nhãn)']}")

    if row["Cosine (3 nhãn)"] > row["Cosine (19 nhãn)"]:
        print("Độ tương đồng giảm khi xét đầy đủ nhãn.")
    elif row["Cosine (3 nhãn)"] < row["Cosine (19 nhãn)"]:
        print("Độ tương đồng tăng khi xét đầy đủ nhãn.")
    else:
        print("Độ tương đồng không thay đổi.")


Phim: Men in Black (a.k.a. MIB) (1997)
Cosine với 3 nhãn: 0.5774
Cosine với 19 nhãn: 0.2582
Độ tương đồng giảm khi xét đầy đủ nhãn.

Phim: Jurassic Park (1993)
Cosine với 3 nhãn: 0.0
Cosine với 19 nhãn: 0.2236
Độ tương đồng tăng khi xét đầy đủ nhãn.


In [3]:
# yêu cầu 2
import pandas as pd

result = pd.DataFrame({
    "Phim so sánh": [
        "Men in Black (1997)",
        "Jurassic Park (1993)"
    ],
    "Cosine (3 nhãn)": [0.5774, 0.0000],
    "Cosine (19 nhãn)": [0.2582, 0.2582]
})

print("Phim tham chiếu: Toy Story (1995)")
display(result)

result.to_csv(
    "bang_ket_qua_cosine.csv",
    index=False,
    encoding="utf-8-sig"
)

Phim tham chiếu: Toy Story (1995)


,Phim so sánh,Cosine (3 nhãn),Cosine (19 nhãn)
0,Men in Black (1997),0.5774,0.2582
1,Jurassic Park (1993),0.0000,0.2582


In [2]:
# kết quả
import pandas as pd

result = pd.DataFrame({
    "Phim so sánh": [
        "Men in Black (1997)",
        "Jurassic Park (1993)"
    ],
    "Cosine (3 nhãn)": [0.5774, 0.0000],
    "Cosine (19 nhãn)": [0.2582, 0.2582]
})

print("Phim tham chiếu: Toy Story (1995)")
display(result)

result.to_csv(
    "bang_ket_qua_cosine.csv",
    index=False,
    encoding="utf-8-sig"
)

Phim tham chiếu: Toy Story (1995)


,Phim so sánh,Cosine (3 nhãn),Cosine (19 nhãn)
0,Men in Black (1997),0.5774,0.2582
1,Jurassic Park (1993),0.0000,0.2582


Kết quả cho thấy độ tương đồng cosine phụ thuộc vào cách lựa chọn các nhãn thể loại. Với 3 nhãn Action, Comedy và Sci-Fi, Men in Black có độ tương đồng 0.5774 với Toy Story do hai phim cùng có nhãn Comedy. Jurassic Park có độ tương đồng bằng 0 vì không có nhãn chung trong ba trục này. Khi sử dụng đầy đủ 19 nhãn, độ tương đồng của Men in Black và Jurassic Park đều bằng 0.2582 vì mỗi phim có một nhãn chung với Toy Story. Điều này cho thấy việc lựa chọn đặc trưng ảnh hưởng đến kết quả so sánh. Tuy nhiên, độ tương đồng cosine chỉ phản ánh sự giống nhau về metadata thể loại, không thể khẳng định mức độ giống nhau về nội dung hoặc sở thích của người xem.